<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/09_time_series_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time Series Analysis

In [1]:
# !pip install yfinance

import datetime as dt

import polars as pl
import pandas as pd
import yfinance as yf

import altair as alt


## Data Wrangling and Exploratory Data Analysis

In [2]:
df_missing = pl.DataFrame({
    "date": [dt.date(2024, 1, 1), dt.date(2024, 1, 2), dt.date(2024, 1, 3)],
    "value": [10, None, 15]
})

df = (
    df_missing
    .with_columns(
        pl.col("value")
        .fill_null(strategy="forward")
        .alias("value_forward")
        )
    .with_columns(
        pl.col("value")
        .fill_null(strategy="backward")
        .alias("value_backward")
    )
    .with_columns(
        pl.col("value")
        .fill_null(strategy="mean")
        .alias("value_mean")
    )
)
df

date,value,value_forward,value_backward,value_mean
date,i64,i64,i64,i64
2024-01-01,10,10,10,10
2024-01-02,null,10,15,12
2024-01-03,15,15,15,15


In [3]:
# Example DataFrame with daily data
df_daily = pl.DataFrame({
    "date": pl.date_range(dt.date(2024, 1, 1), dt.date(2024, 2, 15), "1d", eager=True),
    "sales": range(46)
})

# Downsample to monthly average sales
df_monthly = df_daily.group_by_dynamic(
    "date", every="1mo"
).agg(
    pl.col("sales").mean().alias("average_sales")
)
display(df_monthly)

date,average_sales
date,f64
2024-01-01,15.0
2024-02-01,38.0


In [4]:
# Define the ticker symbol and date range
ticker = 'GOOG'
start_date = '2019-01-01'
end_date = '2025-06-01'

# Download the data using yfinance
df = yf.download(ticker, start=start_date, end=end_date)

# Display the first few rows of the DataFrame
print(df.head())
df_pl = pl.from_pandas(df, include_index=True )
df_pl.columns

/tmp/ipython-input-4-2438495482.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open    Volume
Ticker           GOOG       GOOG       GOOG       GOOG      GOOG
Date                                                            
2019-01-02  51.983501  52.305091  50.485406  50.528152  30652000
2019-01-03  50.502808  52.536715  50.403893  51.742437  36822000
2019-01-04  53.219158  53.225620  51.067348  51.324422  41878000
2019-01-07  53.103840  53.382685  52.426367  53.258423  39638000
2019-01-08  53.496010  53.907565  52.713165  53.487561  35298000


['Date',
 "('Close', 'GOOG')",
 "('High', 'GOOG')",
 "('Low', 'GOOG')",
 "('Open', 'GOOG')",
 "('Volume', 'GOOG')"]

In [5]:
def colname(col: str):
  if col.startswith('('):
        return ( col
          .lstrip('(')
          .rstrip(')')
          .split(",")[0]
          .strip("'")
          .lower()
        )
  return col.lower()

print([colname(col) for col in df_pl.columns])

df_pl = df_pl.rename(colname)
df_pl.head()

['date', 'close', 'high', 'low', 'open', 'volume']


date,close,high,low,open,volume
datetime[ns],f64,f64,f64,f64,i64
2019-01-02 00:00:00,51.983501,52.305091,50.485406,50.528152,30652000
2019-01-03 00:00:00,50.502808,52.536715,50.403893,51.742437,36822000
2019-01-04 00:00:00,53.219158,53.22562,51.067348,51.324422,41878000
2019-01-07 00:00:00,53.10384,53.382685,52.426367,53.258423,39638000
2019-01-08 00:00:00,53.49601,53.907565,52.713165,53.487561,35298000


In [6]:
alt.Chart(df_pl).mark_line().encode(
    x='date',
    y='close',
    tooltip=['date', 'close']
).properties(
    width=800,
    height=400
)

alt.Chart(...)

In [7]:
alt.Chart(
    df_pl.filter(
         pl.col("date").is_between(
             dt.datetime(2020, 1, 1),
             dt.datetime(2022, 12, 31)
             )
    )
    ).mark_line().encode(
    x='date',
    y='close',
    tooltip=['date', 'close']
).properties(
    width=800,
    height=400
)

alt.Chart(...)

In [8]:
df_pl = df_pl.sort('date')

In [9]:
price_monthly = (
    df_pl
    .group_by_dynamic("date", every="1m", period="3m")
    .agg( pl.col("close").mean() )
    .with_columns(
        pl.col("date").dt.strftime("%Y-%m").alias("date")
    )
)
display(price_monthly.head())
alt.Chart(price_monthly).mark_line().encode(
    x='date:T',
    y='close',
    tooltip=['date', 'close']
# ).properties(
#     width=800,
#     height=400
)

date,close
str,f64
"""2019-01""",51.983501
"""2019-01""",50.502808
"""2019-01""",50.502808
"""2019-01""",50.502808
"""2019-01""",53.219158


alt.Chart(...)